# Vulnerability-Aware Java Model — Before vs. After

**FINETUNING_001 (Track 3).** We fine-tuned `Qwen/Qwen2.5-Coder-7B-Instruct`
with a LoRA adapter on labelled Java vulnerability → remediation pairs
(CWE-89 SQL injection, CWE-22 path traversal, CWE-78 command injection), and
prove the gain with an **objective** before/after metric: every fix is checked
by `javac` (does it compile?) and `semgrep` (is the vulnerability gone?).

One vLLM server exposes **both** models — the base and the LoRA adapter
(`vuln-fixer`) — so the comparison is apples-to-apples: identical prompts,
identical scorer, same server.

> **Prereqs:** vLLM is serving in a separate terminal, and the full eval has
> been run once (writes `eval/results/*_results.jsonl`). See the last section
> if a cell complains those files are missing.

In [ ]:
import sys, json
from pathlib import Path

# Make the `finetune` package importable whether this runs from the repo root
# or from inside finetune/.
HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "finetune").is_dir() else HERE.parent
sys.path.insert(0, str(ROOT))

from finetune.data.prepare_dataset import SYSTEM_PROMPT, build_user_prompt, CWE_DESCRIPTIONS
from finetune.eval import run_eval
from finetune.eval.harness import score_patch, extract_java_code

# vLLM endpoint (started separately). --enable-lora exposes both model ids.
ENDPOINT    = "http://localhost:8000/v1"
BASE_MODEL  = "Qwen/Qwen2.5-Coder-7B-Instruct"   # the base model
TUNED_MODEL = "vuln-fixer"                        # the LoRA adapter name

base_client  = run_eval.make_client(f"openai:{ENDPOINT}#{BASE_MODEL}")
tuned_client = run_eval.make_client(f"openai:{ENDPOINT}#{TUNED_MODEL}")
RESULTS_DIR  = ROOT / "finetune" / "eval" / "results"
print("base :", base_client.name)
print("tuned:", tuned_client.name)

## 1. Live demo — fix a vulnerable snippet, base vs. tuned

Pick any vulnerable Java method, send it to **both** models with the *same*
prompt, and let the harness judge each output. The badges are the proof:
`compiles` (javac) and `vuln-fixed` (semgrep).

In [ ]:
import html
from IPython.display import HTML, display

VERDICT = {True: "&#9989;", False: "&#10060;", None: "&mdash;"}  # check / cross / dash

def fix_one(client, vulnerable_code, cwe):
    user = build_user_prompt(vulnerable_code, cwe)
    resp = client.generate(SYSTEM_PROMPT, user, {"vulnerable_code": vulnerable_code, "output": ""})
    scored = score_patch(resp, cwe)
    return (extract_java_code(resp) or resp), scored

def _col(title, code_text, scored):
    badges = (f"format {VERDICT[scored['format_ok']]} &nbsp; "
              f"compiles {VERDICT[scored['compiles']]} &nbsp; "
              f"vuln-fixed {VERDICT[scored['vuln_fixed']]}")
    return ("<td style='vertical-align:top;width:50%;padding:6px'>"
            f"<h4 style='margin:4px 0'>{title}</h4><div>{badges}</div>"
            "<pre style='white-space:pre-wrap;background:#f6f8fa;padding:8px;"
            f"border-radius:6px'>{html.escape(code_text)}</pre></td>")

def compare(vulnerable_code, cwe):
    cwe = cwe if str(cwe).startswith("CWE-") else f"CWE-{cwe}"
    print(f"Input: {cwe} - {CWE_DESCRIPTIONS[cwe]}")
    b_code, b = fix_one(base_client, vulnerable_code, cwe)
    t_code, t = fix_one(tuned_client, vulnerable_code, cwe)
    display(HTML("<table style='width:100%;border-collapse:collapse'><tr>"
                 + _col(f"BASE - {BASE_MODEL}", b_code, b)
                 + _col("TUNED - fine-tuned adapter", t_code, t)
                 + "</tr></table>"))
    return b, t

In [ ]:
SAMPLES = {
    "CWE-89": '''import java.sql.*;

public class UserDao {
    public ResultSet findUser(Connection conn, String userId) throws SQLException {
        Statement stmt = conn.createStatement();
        String query = "SELECT * FROM users WHERE id = '" + userId + "'";
        return stmt.executeQuery(query);
    }
}''',
    "CWE-78": '''import java.io.IOException;

public class Pinger {
    public Process ping(String host) throws IOException {
        return Runtime.getRuntime().exec("ping -c 1 " + host);
    }
}''',
    "CWE-22": '''import java.io.*;

public class ReportReader {
    private static final String BASE = "/srv/reports/";
    public String read(String name) throws IOException {
        BufferedReader r = new BufferedReader(new FileReader(new File(BASE + name)));
        try { return r.readLine(); } finally { r.close(); }
    }
}''',
}

compare(SAMPLES["CWE-89"], "CWE-89");

### Try your own
Edit `MY_CODE` / `MY_CWE` below (CWE-89, CWE-78, or CWE-22) and run the cell.

In [ ]:
MY_CODE = SAMPLES["CWE-78"]
MY_CWE  = "CWE-78"
compare(MY_CODE, MY_CWE);

In [ ]:
# Optional interactive widget (if ipywidgets is installed)
try:
    import ipywidgets as W
    code_box = W.Textarea(value=SAMPLES["CWE-89"], layout=W.Layout(width="100%", height="200px"))
    cwe_dd   = W.Dropdown(options=["CWE-89", "CWE-78", "CWE-22"], value="CWE-89", description="CWE:")
    out, btn = W.Output(), W.Button(description="Fix it: base vs tuned", button_style="primary")
    def _go(_):
        out.clear_output()
        with out:
            compare(code_box.value, cwe_dd.value)
    btn.on_click(_go)
    display(W.VBox([cwe_dd, code_box, btn, out]))
except Exception as e:
    print("ipywidgets not available - use the compare() cell above instead.", e)

## 2. Aggregate before/after metrics (full test set)

Loaded from the eval the runner already produced. This is the deliverable:
**tuned should beat base on compile rate and vuln-fixed rate.**

In [ ]:
import pandas as pd

def load_side(side):
    path = RESULTS_DIR / f"{side}_results.jsonl"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run the full eval once:\n"
            f"  python finetune/eval/run_eval.py \\\n"
            f"    --base-spec  openai:{ENDPOINT}#{BASE_MODEL} \\\n"
            f"    --tuned-spec openai:{ENDPOINT}#{TUNED_MODEL}")
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]

base_res, tuned_res = load_side("base"), load_side("tuned")
base_agg, tuned_agg = run_eval.aggregate(base_res), run_eval.aggregate(tuned_res)

def cell(rate):
    if rate is None:
        return "n/a"
    p, n = rate
    return f"{100 * p / n:.0f}% ({p}/{n})"

print(f"Test items: {base_agg['n']}")
pd.DataFrame(
    {"Base":  [cell(base_agg["format"]),  cell(base_agg["compile"]),  cell(base_agg["vuln_fixed"]),  f"{base_agg['mean_score']:.3f}"],
     "Tuned": [cell(tuned_agg["format"]), cell(tuned_agg["compile"]), cell(tuned_agg["vuln_fixed"]), f"{tuned_agg['mean_score']:.3f}"]},
    index=["Format rate", "Compile rate", "Vuln-fixed rate", "Mean score"])

In [ ]:
# Vuln-fixed rate per CWE
cwes = sorted(set(base_agg["per_cwe_vuln_fixed"]) | set(tuned_agg["per_cwe_vuln_fixed"]))
pd.DataFrame(
    {"Base":  [cell(base_agg["per_cwe_vuln_fixed"].get(c)) for c in cwes],
     "Tuned": [cell(tuned_agg["per_cwe_vuln_fixed"].get(c)) for c in cwes]},
    index=cwes)

## 3. Where the tuned model wins

Cases from the test set where the **tuned** model removed the vulnerability and
the **base** model did not - the qualitative side of the metric.

In [ ]:
by_id = {r["source_id"]: r for r in base_res}
wins = [t for t in tuned_res
        if t["vuln_fixed"] and by_id.get(t["source_id"], {}).get("vuln_fixed") is False]
print(f"{len(wins)} case(s) where TUNED fixed and BASE did not (of {len(tuned_res)} test items)")

for t in wins[:3]:
    b = by_id[t["source_id"]]
    display(HTML(
        f"<h4>{html.escape(str(t['source_id']))} &mdash; {t['cwe']}</h4>"
        "<table style='width:100%;border-collapse:collapse'><tr>"
        + _col("BASE output", extract_java_code(b["response"]) or b["response"], b)
        + _col("TUNED output", extract_java_code(t["response"]) or t["response"], t)
        + "</tr></table>"))

## Summary

- **Same base model, same prompts, same objective scorer** - the only variable
  is the LoRA adapter.
- The adapter improves **compile rate** and **vuln-fixed rate** over the base,
  proving fine-tuning produces more accurate, policy-aligned Java fixes than the
  base model.

**Future work:** add a prompted-baseline column (base + strong system prompt) to
show fine-tuning beats prompting at lower token cost; extend to more CWEs; serve
the adapter as a specialist fixer inside an incident-response agent.